# 3.4 Drug Discovery Agent with MCP + Claude Agent SDK

In chapters 3.2 and 3.3 we saw how to write boilerplate code and implement agents. But the problem is, if you are not a python programmer you mind find this to be tedious. As a scientist this may not be the problem you want to solve. 

## 3.4.1 Claude Agent SDK

This is where **Claude Agent SDK** comes into play. This library developed by Anthropic helps anyone with minimal programming knowledge to implement agents or agent teams easily.

**Advantages of SDK**

To help you understand a few advantages of Claude Agent SDK, in the previous chapters we saw that when we implement the code for the agent, we have to write `if/else` statements to execute each tool and feed the result back to the agent. But the SDK takes care of everything under the hood. The SDK also comes with a few built-in tools such as `read`, `write`, `websearch`. Most importantly, the SDK can be used to spawn and orchestrate multiple subagents. 

**Trade-offs**

- It can get expensive quickly
- Runs as a subprocess in your local computer or a cloud computer
- Developed and maintained by Anthropic

## 3.4.2 Model Context Protocol (MCP) Servers

Another concept you have to learn before you deploy an agent with Claude SDK is **MCP Servers**. You can think of an MCP server as an *advanced version* of a tool. 
MCP is an open standard for connecting AI agents to external systems.

A tool is a single callable function with a name, description, input schema and some logic that runs and returns a result. For example `analyze_protein_sequence`, `summarize_protein_role` from previous chapters are tools. 

On the other hand, an MCP server is a *container* that hosts and exposes one or more tools over a standardized protocol. It allows the agent to discovery what tools it has rather than listing each tool one by one, executes the tool and return the results in a standard format. Again, a tool is a function, but an MCP server is an API service that exposes functions over a protocol. 

With raw tool use (your original notebook), you wire everything manually; you write the JSON schema, you write the `if/elif` dispatch, and all of it lives in your agent code. It works, but it doesn't travel well. With MCP, the tools live in the server, which is a separate, reusable process. Any MCP-compatible client (Claude, Cursor, your own agent, someone else's agent) can connect to it and immediately discover and use those tools — without you writing any glue code. The server is the shareable, deployable unit.

## 3.4.3 What changes from the previous approach?

| | Previous (native tool use) | This notebook (MCP) |
|---|---|---|
| Tools defined | Inline Python functions + JSON schema | MCP Server that advertises tools automatically |
| Tool execution | Manual `if/elif` dispatch | MCP protocol handles dispatch |
| Reusability | Single notebook | Server can be reused across any MCP-compatible client |
| Multi-tool scaling | Grows complex fast | Plug-and-play: add a new server, get new tools |


## 3.4.4 Implement an agent with Claude Agent SDK + MCP Servers

### Step 1 — Install dependencies

In [ ]:
!uv pip install anthropic biopython mcp

### Step 2 — API key helper

In [ ]:
import os
from dotenv import load_dotenv

def get_api_key():
    """Load Anthropic API key from Colab secrets or environment variable."""
    try:
        from google.colab import userdata
        return userdata.get("ANTHROPIC_API_KEY")
    except ImportError:
        load_dotenv()
        api_key = os.environ.get("ANTHROPIC_API_KEY")
        if not api_key:
            raise ValueError(
                "API key not found. Set ANTHROPIC_API_KEY env var or add it to .env"
            )
        return api_key

print("API key helper ready.")

### Step 3 — Write the MCP Server to disk

An MCP Server is a **standalone Python script** that:
1. Declares tools using the `@mcp.tool()` decorator (no manual JSON schema!)
2. Runs as a subprocess that Claude connects to over `stdio`

Here's the catch: We have to write the server file to disk so it can be launched as a separate process. Run the following cell to write the python script to a file.

In [ ]:
server_code = '''
#!/usr/bin/env python
"""
bio_tools_server.py — MCP Server exposing two bioinformatics tools:
  1. analyze_protein_sequence  — biophysical properties via BioPython
  2. summarize_protein_role    — biological context summary via Claude
"""
import os
import anthropic
from mcp.server.fastmcp import FastMCP
from Bio.SeqUtils.ProtParam import ProteinAnalysis

# FastMCP auto-generates the JSON schema from type hints and docstrings.
# No manual schema writing needed!
mcp = FastMCP("bio_tools")

client = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])


@mcp.tool()
def analyze_protein_sequence(sequence: str) -> dict:
    """
    Analyze a protein amino acid sequence and return biophysical properties.

    Args:
        sequence: Protein sequence in one-letter amino acid code (e.g. \"MKTII...\")

    Returns:
        Dictionary with length, molecular_weight_Da, isoelectric_point,
        instability_index, gravy_score, amino_acid_percent, is_stable,
        and is_hydrophilic.
    """
    sequence = sequence.upper().strip()
    analysis = ProteinAnalysis(sequence)
    results = {
        "length":              len(sequence),
        "molecular_weight_Da": round(analysis.molecular_weight(), 2),
        "isoelectric_point":   round(analysis.isoelectric_point(), 2),
        "instability_index":   round(analysis.instability_index(), 2),
        "gravy_score":         round(analysis.gravy(), 3),
        "amino_acid_percent":  {
            aa: round(pct, 1)
            for aa, pct in analysis.get_amino_acids_percent().items()
            if pct > 0
        },
    }
    results["is_stable"]      = results["instability_index"] < 40
    results["is_hydrophilic"] = results["gravy_score"] < 0
    return results


@mcp.tool()
def summarize_protein_role(protein_name: str, organism: str) -> dict:
    """
    Summarize the biological role, disease association, and drug-target
    relevance of a named protein.

    Args:
        protein_name: Common name or gene symbol (e.g. \"EGFR\", \"p53\")
        organism:     The organism (e.g. \"human\", \"mouse\")

    Returns:
        Dictionary with a \"summary\" key containing a 3-4 sentence description.
    """
    response = client.messages.create(
        model="claude-opus-4-5",
        max_tokens=400,
        messages=[{
            "role": "user",
            "content": (
                f"Provide a concise 3–4 sentence summary of the protein \'{protein_name}\' "
                f"in {organism}. Cover: (1) its biological function, "
                f"(2) which disease or condition it is associated with, "
                f"(3) why it is considered a drug target. Be factual and concise."
            )
        }]
    )
    return {"summary": response.content[0].text}


if __name__ == "__main__":
    # Run with stdio transport — Claude will talk to us over stdin/stdout
    mcp.run(transport="stdio")
'''

with open("bio_tools_server.py", "w") as f:
    f.write(server_code)

print("✅ bio_tools_server.py written to disk.")

### Step 4 — Build the MCP Client + Agent Loop

The Claude Agent SDK (`anthropic`) includes a built-in MCP client.

Key differences from the raw tool-use approach:
- **`StdioServerParameters`** — tells Claude how to launch our server script
- **`client.beta.messages.create(..., mcp_servers=[...])`** — Claude auto-discovers all tools from the server; no manual `tools=[]` list needed
- The dispatch `if/elif` block is gone — MCP handles routing automatically

In [ ]:
import anthropic
from anthropic import Anthropic

client = Anthropic(api_key=get_api_key())

print("Anthropic client ready.")

In [ ]:
import subprocess, sys, os

# MCP server configuration — Claude will launch this as a subprocess
# and communicate via stdio (stdin/stdout)
mcp_server_config = {
    "type": "stdio",
    "command": sys.executable,          # same Python interpreter as this notebook
    "args": ["bio_tools_server.py"],
    "env": {"ANTHROPIC_API_KEY": get_api_key()}  # pass the key into the server process
}

SYSTEM_PROMPT = """You are a computational biology assistant specializing in drug discovery.
When given a protein of interest, you:
1. Use analyze_protein_sequence to compute its biophysical properties.
2. Use summarize_protein_role to understand its biological context.
3. Combine both results to generate 2–3 specific, testable drug discovery hypotheses.

Always explain your reasoning in plain language suitable for a bench biologist."""


def run_agent_mcp(user_question: str, verbose: bool = True) -> str:
    """
    Runs the drug discovery agent using MCP for tool discovery and execution.

    Claude auto-discovers tools from bio_tools_server.py — no manual tool
    definitions or dispatch logic required here.
    """
    messages = [{"role": "user", "content": user_question}]

    if verbose:
        print(f"🧪 Question: {user_question[:120]}...\n")
        print("─" * 60)

    # Agent loop — same structure as before, but Claude handles tool routing
    while True:
        response = client.beta.messages.create(
            model="claude-opus-4-5",
            max_tokens=1500,
            system=SYSTEM_PROMPT,
            messages=messages,
            # 🔑 MCP magic: Claude discovers and calls tools from the server automatically
            mcp_servers=[mcp_server_config],
            betas=["mcp-client-2025-04-04"],
        )

        if response.stop_reason == "end_turn":
            # Grab the final text block
            final_answer = next(
                (b.text for b in response.content if hasattr(b, "text")), ""
            )
            if verbose:
                print("─" * 60)
                print("🤖 Agent's Final Answer:\n")
                print(final_answer)
            return final_answer

        elif response.stop_reason in ("tool_use", "mcp_tool_use"):
            # Show what tools were called (verbose mode)
            if verbose:
                for block in response.content:
                    if hasattr(block, "name"):   # tool call block
                        print(f"🔧 MCP tool called: {block.name}")
                        if hasattr(block, "input"):
                            print(f"   Inputs: {block.input}")

            # Append assistant turn and continue — MCP handles result injection
            messages.append({"role": "assistant", "content": response.content})

        else:
            print(f"⚠️  Unexpected stop reason: {response.stop_reason}")
            break


print("Agent function defined.")

Did you understand what's happening with the following code block? 

```
elif response.stop_reason in ("tool_use", "mcp_tool_use"):
            # Show what tools were called (verbose mode)
            if verbose:
                for block in response.content:
                    if hasattr(block, "name"):   # tool call block
                        print(f"🔧 MCP tool called: {block.name}")
                        if hasattr(block, "input"):
                            print(f"   Inputs: {block.input}")

            # Append assistant turn and continue — MCP handles result injection
            messages.append({"role": "assistant", "content": response.content})
```

This is somewhat similar to what we saw in our previous chapter, where we  manually run the function, then manually send the result back to Claude.

```
if tool_name == "analyze_protein_sequence":
    result = analyze_protein_sequence(**tool_inputs)  # ← execution here
```

But the amazing advantage of using the Claude Agent SDK is that the tool execution happened even before we hit this `elif` block. 

When you called `client.beta.messages.create(..., mcp_servers=[...])`, the SDK:

1. Got Claude's tool call request
2. Forwarded it to bio_tools_server.py over stdio
3. Got the result back
4. Injected the result into response.content
5. Then returned the response to your code

So by the time your code sees `stop_reason = "mcp_tool_use"`, the tool has already run. That `elif` block is really just printing what already happened.

To summarize what happens,

```
client.beta.messages.create(...)
    │
    ├── Claude decides: "call analyze_protein_sequence"
    ├── SDK forwards request to bio_tools_server.py over stdio
    ├── bio_tools_server.py runs the function
    ├── SDK gets the result back
    │
    └── returns to your code   ← you're already past the execution
            │
            └── your elif block just logs + appends for the next turn
```

### Step 5 — Run the agent

Same EGFR question as before — but now Claude is talking to the **MCP server** to discover and execute the tools.

In [ ]:
egfr_sequence_fragment = (
    "MRPSGTAGAALLALLAALCPASRALEEKKVCQGTSNKLTQLGTFEDHFLSLQRMFNNCEVVLGNLEITYVQRNYDLSFLKTIQ"
    "EVAGYVLIALNTVERIPLENLQIIRGNMYYENSYALAVLSNYDANKTGLKELPMRNLQEILHGAVRFSNNPALCNVESIQWRD"
    "IVSSDFLSNMSMDFQNHLGSCQKCDPSCPNGSCWGAGEENCQKLTKIICAQQCSGRCRGKSPSDCCHNQCAAGCTGPRESDCL"
    "VCRKFRDEATCKDTCPPLMLYNPTTYQMDVNPEGKYSFGATCVKKCPRNYVVTDHGSCVRACGADSYEMEEDGVRKCKKCEG"
    "PCRKVCNGIGIGEFKDSLSINATNIKHFKNCTSISGDLHILPVAFRGDSFTHTPPLDPQELDILKTVKEITGFLLIQAWPENR"
)

question = f"""
I'm studying the role of EGFR in non-small-cell lung cancer.
Here is a fragment of its amino acid sequence:

{egfr_sequence_fragment}

Please analyze this protein and help me think about potential
small-molecule drug targeting strategies.
"""

answer = run_agent_mcp(question, verbose=True)

### Scaling up: adding more tools

The power of MCP is that adding new tools requires **only editing the server**.
The agent loop above stays unchanged.

Here's how you'd add a PubMed search tool — add this to `bio_tools_server.py`:

```python
import urllib.request, json

@mcp.tool()
def search_pubmed(query: str, max_results: int = 5) -> dict:
    """
    Search PubMed for recent papers about a protein or drug target.

    Args:
        query:       Search term, e.g. "EGFR inhibitors lung cancer"
        max_results: Number of results to return (default 5)
    """
    base = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/"
    search_url = f"{base}esearch.fcgi?db=pubmed&term={urllib.parse.quote(query)}&retmax={max_results}&retmode=json"
    with urllib.request.urlopen(search_url) as r:
        ids = json.load(r)["esearchresult"]["idlist"]
    return {"pubmed_ids": ids, "query": query}
```

After saving, Claude will discover and use `search_pubmed` **automatically** on the next run — no changes to your agent loop needed.

### Claude Skills

While you can write as many MCP servers as you'd want, you can also use already built MCP servers called **Claude skills**. These are usually built by Anthropic or the community. So, instead of writing your own `bio_tools_server.py`, someone already wrote a "PubMed skill" or "UniProt skill" that you just plug in. The "portable" part means the same skill can be reused across different agents and projects — you don't rebuild it every time.

:::{admonition} 
:class: tip
Before skills: you write every tool yourself
After skills:  you plug in pre-built capability packages
:::

### Summary

| Step | What happened |
|------|---------------|
| 1 | Wrote `bio_tools_server.py` — an MCP server with two tools |
| 2 | Claude launched the server as a subprocess via `stdio` transport |
| 3 | Claude auto-discovered tool schemas — no manual JSON schema |
| 4 | Claude called tools via MCP; dispatch was handled by the protocol |
| 5 | Claude synthesized results into drug discovery hypotheses |

**MCP** separates tool logic (the server) from agent logic (the loop),
making each independently reusable, testable, and shareable.